## Import PyTerrier

In [9]:

import os
os.environ["JAVA_HOME"] = "C:\Program Files\Java\jdk-25.0.2"
#os.environ["JVM_PATH"] = "C:\Program Files\Java\jdk-25.0.2\lib\jvm.lib"

import pyterrier as pt
#pt.java.init()

print(" PyTerrier initialized successfully with local JDK.")

import os
import json
import pandas as pd
import pyterrier as pt

 PyTerrier initialized successfully with local JDK.


<>:2: SyntaxWarning: invalid escape sequence '\P'
<>:2: SyntaxWarning: invalid escape sequence '\P'
C:\Users\krist\AppData\Local\Temp\ipykernel_14596\2539851376.py:2: SyntaxWarning: invalid escape sequence '\P'
  os.environ["JAVA_HOME"] = "C:\Program Files\Java\jdk-25.0.2"


# Load Data

In [10]:
# Load data
MPD_PATH = "./mpd_data"

def load_mpd_playlists(path):
    df = []

    for filename in os.listdir(path):
        with open(os.path.join(path, filename), "r", encoding="utf-8") as file:
            data = json.load(file)

            for playlist in data["playlists"]:
                docno = str(playlist["pid"])
                title = playlist.get("name", "")

                tracks = playlist.get("tracks", "") # "" as a default value
                track_text = ""
                for track in tracks: 
                    track_name = track.get("track_name", "")
                    artist_name = track.get("artist_name", "")
                    album_name = track.get("album_name", "")
                    track_name = (f"{track_name} {artist_name} {album_name}")

                full_text = title + " " + " ".join(track_text)

                df.append({
                    "docno": docno,
                    "text": full_text, 
                    "songs": tracks
                })

    return pd.DataFrame(df)

print("Loading playlists...")
df = load_mpd_playlists(MPD_PATH)
print(f"Loaded {len(df)} playlists")


Loading playlists...
Loaded 3000 playlists


In [11]:
print(df.sample(n=10))

     docno                      text  \
1819  1819                      Mom    
1964  1964                   NewNew    
1479  1479                    peter    
2860  2860  Smells Like Teen Spirit    
2618  2618            shower songs     
1981  1981                 outside     
2709  2709                       Ok    
1020  1020                vaporwave    
172    172            motivational     
437    437                    April    

                                                  songs  
1819  [{'pos': 0, 'artist_name': 'Antonín Dvořák', '...  
1964  [{'pos': 0, 'artist_name': 'LoveRance', 'track...  
1479  [{'pos': 0, 'artist_name': 'Sam Smith', 'track...  
2860  [{'pos': 0, 'artist_name': 'Nirvana', 'track_u...  
2618  [{'pos': 0, 'artist_name': 'Amanda Cook', 'tra...  
1981  [{'pos': 0, 'artist_name': 'MKTO', 'track_uri'...  
2709  [{'pos': 0, 'artist_name': 'Bryson Tiller', 't...  
1020  [{'pos': 0, 'artist_name': 'Blank Banshee', 't...  
172   [{'pos': 0, 'artist_name': 'Kel

## Build Index

In [12]:
base = os.path.abspath("var/index") # Append new folders onto absolute path
os.makedirs(base, exist_ok=True) 

indexer = pt.index.IterDictIndexer(base, overwrite=True)
index_ref = indexer.index(df.to_dict(orient="records"))

17:46:36.850 [main] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (23) - further warnings are suppressed
17:46:38.572 [main] WARN org.terrier.structures.indexing.Indexer -- Indexed 124 empty documents
17:46:38.580 [main] ERROR org.terrier.structures.indexing.Indexer -- Could not rename index
java.io.IOException: Rename of index structure file 'c:\Users\krist\Documents\UW\2025-2026\INFO376\final-project\376-final-project\var\index/data_1.direct.bf' (exists) to 'c:\Users\krist\Documents\UW\2025-2026\INFO376\final-project\376-final-project\var\index/data.direct.bf' (exists) failed - likely that source file is still open. Possible indexing bug?
	at org.terrier.structures.IndexUtil.renameIndex(IndexUtil.java:379)
	at org.terrier.structures.indexing.Indexer.index(Indexer.java:388)


In [13]:
# Explore Index
index = pt.IndexFactory.of(index_ref)

print("Number of documents:", index.getCollectionStatistics().getNumberOfDocuments())
print("Number of unique terms:", index.getCollectionStatistics().getNumberOfUniqueTerms())
print("Number of tokens:", index.getCollectionStatistics().getNumberOfTokens())
print("Average doc length:", index.getCollectionStatistics().getAverageDocumentLength())


Number of documents: 1000
Number of unique terms: 18441
Number of tokens: 268505
Average doc length: 268.505


In [14]:
# Peek at first 20 terms in the index
it = index.getLexicon()
for i, (term, entry) in enumerate(it):
    if i >= 40:
        break
    print(term, "-> df:", entry.getDocumentFrequency(), "cf:", entry.getFrequency())


0 -> df: 42 cf: 43
00 -> df: 1 cf: 1
000 -> df: 24 cf: 29
01 -> df: 6 cf: 6
02 -> df: 6 cf: 6
03 -> df: 7 cf: 7
05 -> df: 4 cf: 32
06 -> df: 6 cf: 8
07 -> df: 10 cf: 12
08 -> df: 1 cf: 1
09 -> df: 3 cf: 3
0scill8or -> df: 1 cf: 1
1 -> df: 163 cf: 232
10 -> df: 43 cf: 86
100 -> df: 49 cf: 53
1000 -> df: 6 cf: 6
1000hp -> df: 1 cf: 1
1000volt -> df: 1 cf: 1
1000x -> df: 4 cf: 4
1007 -> df: 3 cf: 3
100k -> df: 1 cf: 1
100x -> df: 1 cf: 2
101 -> df: 8 cf: 15
103rd -> df: 1 cf: 1
1041 -> df: 1 cf: 1
1042 -> df: 1 cf: 1
1043 -> df: 1 cf: 1
1046 -> df: 1 cf: 1
1048 -> df: 1 cf: 2
1049 -> df: 1 cf: 1
1050 -> df: 1 cf: 1
1056r -> df: 1 cf: 1
1067 -> df: 2 cf: 4
1068 -> df: 2 cf: 2
10cc -> df: 6 cf: 7
10th -> df: 2 cf: 2
11 -> df: 32 cf: 36
110th -> df: 2 cf: 2
112 -> df: 13 cf: 25
113 -> df: 1 cf: 1


Make a search

In [15]:
query = "late night"
bm25 = pt.terrier.Retriever(index, wmodel="BM25")
results = bm25.search(query)
print(results)

    qid  docid docno  rank     score       query
0     1    198   198     0  6.378549  late night
1     1    759   759     1  6.271693  late night
2     1    486   486     2  6.134824  late night
3     1    809   809     3  5.902465  late night
4     1    976   976     4  5.889344  late night
..   ..    ...   ...   ...       ...         ...
411   1    406   406   411  0.354850  late night
412   1    847   847   412  0.353629  late night
413   1    264   264   413  0.336553  late night
414   1    843   843   414  0.328750  late night
415   1     72    72   415  0.326921  late night

[416 rows x 6 columns]


In [16]:
for row in df.iloc[506].songs:
    print(f"{row['track_name']} by {row['album_name']}")
    #print(row)


Dile Que Tu Me Quieres by Odisea
Bebe (feat. Anuel AA) by Bebe (feat. Anuel AA)
Tú Foto by Odisea
Ahora Dice by Ahora Dice
Hey DJ by Hey DJ
Reggaetón Lento (Bailemos) by Primera Cita
Tan Fácil by Primera Cita
Para Enamorarte by Primera Cita
Primera Cita by Primera Cita
No Entiendo by Primera Cita
El Amante by Fénix
Si Tú La Ves by Fénix
Hasta el Amanecer by Fénix
El Perdón by Fénix
Bebé - Remix by Bebé
Te Busco (feat. Nicky Jam) by Blanco Perla
Danza Kuduro by Meet The Orphans
Sr. Destino by Meet The Orphans
La Rompe Corazones by La Rompe Corazones
Gyal You A Party Animal - Remix by Gyal You A Party Animal
Shaky Shaky by Shaky Shaky
Limbo by Prestige
No Me Doy Por Vencido by Palabras Del Silencio
Aqui Estoy Yo by Palabras Del Silencio
Te Quiero Pa´Mi by King Of Kings 10th Anniversary
Encanto by Encanto
Mayor Que Yo 3 by Dance Latin # 1 Hits
Dile by The Last Don
SUBEME LA RADIO REMIX by Trouble
DUELE EL CORAZON by DUELE EL CORAZON
Bailando - Spanish Version by SEX AND LOVE
Cuando Me Ena